# Quantification Pipeline

This notebook runs the per-FOV inventory step and then combines the resulting CSVs for each experimental group.

## Configuration

`INCLUDE` accepts one or more TIFF or ND2 patterns. Labels must be stored beside each source as `<stem>_labels.png`, `<stem>_labels.tif`, or `<stem>_labels.tiff` (PNG takes precedence if more than one is present).

In [ ]:
from pathlib import Path

from src.finalize.main import finalize
from src.quant.batch_inventory import batch_inventory
from src.quant.focus_detection import FocusDetectionConfig

DATA = Path("../../data")
OUT = Path("./out")

INCLUDE = [
    "MCPmTQ/**/*.nd2",
    # Add TIFF patterns explicitly when needed to load TIF files, for example:
    # "example/**/*.tif",
]
EXCLUDE = []

CELLS_DIR = OUT / "cells"
CELL_LAYERS_DIR = OUT / "cell_layers"
FOCI_DIR = OUT / "foci"

/nix/store/vmyzlskqbram80w53g9h5xkfnw9hsppg-pipeline-virtual-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Inventory

This writes one cells CSV, cell layers CSV, and foci CSV for every selected FOV.

In [2]:
FOCUS_DETECTION_CONFIG = FocusDetectionConfig(
    detection_threshold_method="IQR_multiple",
    absolute_detection_threshold=1.0,
    IQR_multiple=2.0,
    min_sigma=2.0,
    max_sigma=3.8,
    num_sigma=20,
    overlap=0.75,
    expand_by=2.0,
    bg_sub_radius=1.5,
    denoise_radius=2.0,
    denoise_amount=6.0,
    gaussian_sigma=1.2,
    plot_crop_margin=3,
)

batch_inventory(
    root=DATA,
    include=INCLUDE,
    exclude=EXCLUDE,
    focus_detection_config=FOCUS_DETECTION_CONFIG,
    cells_dir=CELLS_DIR,
    cell_layers_dir=CELL_LAYERS_DIR,
    foci_dir=FOCI_DIR,
    # Switch the commented out line if you want to save stepwise progress from the foci detection
    stepwise_figures_dir=None,
    # stepwise_figures_dir = OUT / "stepwise",
    max_workers=0,
)

inventory ../../data/MCPmTQ/2026_3_9_JKP21_MCPmTQ/1.nd2: 492cell [00:03, 127.89cell/s]
Batch inventory: 100%|██████████| 1/1 [00:04<00:00,  4.80s/FOV]


## Step 2: Finalize

This combines the per-FOV CSVs across biological replicates within each group and creates the aggregate plots.

In [3]:
finalize(
    root=DATA,
    include=INCLUDE,
    exclude=EXCLUDE,
    out=OUT / "final",
    cells_dir=CELLS_DIR,
    cell_layers_dir=CELL_LAYERS_DIR,
    foci_dir=FOCI_DIR,
    min_bio_reps=3,
    max_workers=8,
    enable_spideymaps=False,
)

Handling groups:   0%|          | 0/1 [00:01<?, ?group/s]


ValueError: expected cells from at least 3 biological replicates (i.e. images from different dates) but found just 1: <StringArray>
['3/9/2026']
Length: 1, dtype: str